In [4]:
"""
Weather + Irrigation Load Script
-----------------------------------------------------
This script:
    - Extracts daily + hourly weather data from Open-Meteo API
    - Drops existing tables (if RESET_TABLES=true)
    - Creates all required tables in SQL Server except for the weather code table that 
    - Transforms API data into clean DataFrames
    - Loads all tables in one execution

Tables created:
    - DimDate
    - FactDailyWeather
    - FactHourlyWeather
    - FactIrrigationDecision 

"""

import os
from datetime import datetime, timedelta

import pandas as pd
import requests
from sqlalchemy import create_engine, text


# Database Connection

connection_string = (
    "mssql+pyodbc://localhost/IrrigationDB?"
    "driver=ODBC+Driver+18+for+SQL+Server"
    "&TrustServerCertificate=yes"
)

engine = create_engine(connection_string, fast_executemany=True)


# Config

RESET_TABLES = True  # Set to False if you do NOT want to drop tables


# Pull weather data from Open-Meteo API


def extract_weather_data(lat, lon):
    today = datetime.today().date()
    yesterday = today - timedelta(days=1)

    start = yesterday.strftime("%Y-%m-%d")
    end = today.strftime("%Y-%m-%d")

    daily_url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}"
        f"&start_date={start}&end_date={end}"
        f"&daily=weathercode,rain_sum"
        f"&timezone=auto"
    )

    hourly_url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}"
        f"&start_date={start}&end_date={end}"
        f"&hourly=temperature_2m,precipitation_probability,rain,"
        f"weathercode,precipitation,showers"
        f"&timezone=auto"
    )

    daily_json = requests.get(daily_url).json()
    hourly_json = requests.get(hourly_url).json()

    daily_df = pd.DataFrame({
        "date_id": daily_json["daily"]["time"],
        "weather_code": daily_json["daily"]["weathercode"],
        "rain_sum": daily_json["daily"]["rain_sum"],
    })

    hourly_df = pd.DataFrame({
        "timestamp": hourly_json["hourly"]["time"],
        "temperature_2m": hourly_json["hourly"]["temperature_2m"],
        "precipitation_probability": hourly_json["hourly"]["precipitation_probability"],
        "rain": hourly_json["hourly"]["rain"],
        "weather_code": hourly_json["hourly"]["weathercode"],
        "precipitation": hourly_json["hourly"]["precipitation"],
        "showers": hourly_json["hourly"]["showers"],
    })

    hourly_df["timestamp"] = pd.to_datetime(hourly_df["timestamp"])
    hourly_df["date_id"] = hourly_df["timestamp"].dt.date.astype(str)

    return daily_df, hourly_df


# Clean + prepare data


def transform_data(daily_df, hourly_df):
    # Remove duplicates
    daily_df = daily_df.drop_duplicates(subset=["date_id"])
    hourly_df = hourly_df.drop_duplicates(subset=["timestamp"])

    # Basic irrigation decision (Week 2 version)
    decisions = []
    for i in range(len(daily_df)):
        today = daily_df.iloc[i]
        yesterday_rain = daily_df.iloc[i - 1]["rain_sum"] if i > 0 else 0

        if today["rain_sum"] > 0.2 or yesterday_rain > 0.2:
            decision = "SKIP"
            reason = "Rain detected"
        else:
            decision = "WATER"
            reason = "Dry conditions"

        decisions.append({
            "date_id": today["date_id"],
            "decision": decision,
            "reason": reason,
            "yesterday_rain": float(yesterday_rain),
            "today_rain": float(today["rain_sum"]),
        })

    irrigation_df = pd.DataFrame(decisions)

    return daily_df, hourly_df, irrigation_df


# Drop tables, create schema, load data

CREATE_SCHEMA_SQL = """

---------------------------------------------------------
-- DROP TABLES (if enabled)
---------------------------------------------------------
IF OBJECT_ID('FactIrrigationDecision', 'U') IS NOT NULL DROP TABLE FactIrrigationDecision;
IF OBJECT_ID('FactHourlyWeather', 'U') IS NOT NULL DROP TABLE FactHourlyWeather;
IF OBJECT_ID('FactDailyWeather', 'U') IS NOT NULL DROP TABLE FactDailyWeather;
IF OBJECT_ID('DimDate', 'U') IS NOT NULL DROP TABLE DimDate;


---------------------------------------------------------
-- CREATE TABLES
---------------------------------------------------------

CREATE TABLE DimDate (
    date_id DATE NOT NULL PRIMARY KEY,
    year INT,
    month INT,
    day INT,
    weekday_name VARCHAR(20)
);

CREATE TABLE FactDailyWeather (
    date_id DATE NOT NULL PRIMARY KEY,
    weather_code INT,
    rain_sum FLOAT,
    FOREIGN KEY (date_id) REFERENCES DimDate(date_id)
);

CREATE TABLE FactHourlyWeather (
    timestamp DATETIME NOT NULL PRIMARY KEY,
    date_id DATE NOT NULL,
    temperature_2m FLOAT,
    precipitation_probability INT,
    rain FLOAT,
    weather_code INT,
    precipitation FLOAT,
    showers FLOAT,
    FOREIGN KEY (date_id) REFERENCES DimDate(date_id)
);

CREATE TABLE FactIrrigationDecision (
    date_id DATE NOT NULL PRIMARY KEY,
    decision VARCHAR(20),
    reason VARCHAR(255),
    yesterday_rain FLOAT,
    today_rain FLOAT,
    FOREIGN KEY (date_id) REFERENCES DimDate(date_id)
);
"""


def load_tables(daily_df, hourly_df, irrigation_df):
    with engine.begin() as conn:
        if RESET_TABLES:
            conn.execute(text(CREATE_SCHEMA_SQL))

        # Load DimDate
        dim_dates = pd.DataFrame({"date_id": pd.concat([daily_df["date_id"], hourly_df["date_id"]]).unique()})
        dim_dates["year"] = pd.to_datetime(dim_dates["date_id"]).dt.year
        dim_dates["month"] = pd.to_datetime(dim_dates["date_id"]).dt.month
        dim_dates["day"] = pd.to_datetime(dim_dates["date_id"]).dt.day
        dim_dates["weekday_name"] = pd.to_datetime(dim_dates["date_id"]).dt.day_name()

        for _, row in dim_dates.iterrows():
            conn.execute(text("""
                INSERT INTO DimDate (date_id, year, month, day, weekday_name)
                VALUES (:date_id, :year, :month, :day, :weekday_name)
            """), row.to_dict())

        # Load FactDailyWeather
        for _, row in daily_df.iterrows():
            conn.execute(text("""
                INSERT INTO FactDailyWeather (date_id, weather_code, rain_sum)
                VALUES (:date_id, :weather_code, :rain_sum)
            """), row.to_dict())

        # Load FactHourlyWeather
        for _, row in hourly_df.iterrows():
            conn.execute(text("""
                INSERT INTO FactHourlyWeather
                (timestamp, date_id, temperature_2m, precipitation_probability,
                 rain, weather_code, precipitation, showers)
                VALUES
                (:timestamp, :date_id, :temperature_2m, :precipitation_probability,
                 :rain, :weather_code, :precipitation, :showers)
            """), row.to_dict())

        # Load FactIrrigationDecision
        for _, row in irrigation_df.iterrows():
            conn.execute(text("""
                INSERT INTO FactIrrigationDecision
                (date_id, decision, reason, yesterday_rain, today_rain)
                VALUES
                (:date_id, :decision, :reason, :yesterday_rain, :today_rain)
            """), row.to_dict())


# Main function call


def run_load_script(lat, lon):
    print("Extracting weather data...")
    daily_df, hourly_df = extract_weather_data(lat, lon)

    print("Transforming data...")
    daily_df, hourly_df, irrigation_df = transform_data(daily_df, hourly_df)

    print("Creating schema + loading tables...")
    load_tables(daily_df, hourly_df, irrigation_df)

    print("Data Load Completed")
  
if __name__ == "__main__":
    run_load_script(38.2527, -85.7585)


Extracting weather data...
Transforming data...
Creating schema + loading tables...


C:\Users\email\anaconda3\Lib\contextlib.py:141: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  return next(self.gen)


Data Load Completed
